## 7c. Flights — AeroDataBox via RapidAPI

Arrivals and departures at London Heathrow via the [AeroDataBox](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/) flights endpoint. You'll need a free RapidAPI key — sign up [here](https://rapidapi.com/auth/sign-up) and subscribe to the Basic (free) tier. The free plan is limited to ~150 requests/month so be resourceful. Feel free to look for alternative endpoints, but settlement will be based on this data provider.

Two query styles are available (max 12h window each):
- **By relative time:** `offset_minutes` + `duration_minutes` relative to now
- **By time range:** explicit local times `fromLocal` / `toLocal` (format: `YYYY-MM-DDTHH:mm`)

The API also supports several boolean filters — check the [AeroDataBox docs](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/playground/apiendpoint_3dbf8f9a-22de-4a99-8e7d-e542f6e63e4f) to learn what's available.

**Relevant for:** LHR_COUNT (total flights in 24h), LHR_INDEX (imbalance metric per 30-min interval).

In [4]:
import requests
import json
import os
import pandas as pd
from datetime import datetime
import time

AERODATABOX_KEY = "954aeb65f0mshc43dfde89d2698bp14c7ddjsnc2bec6059409"  # Replace with your key
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_combined_flights(offsets=[-720, 0, 720], duration=720):
    """
    Fetches flight data for multiple time windows and combines them.
    """
    master_data = {"arrivals": [], "departures": []}
    
    for offset in offsets:
        params = f"?offsetMinutes={offset}&durationMinutes={duration}&direction=Both"
        url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{AIRPORT}{params}"
        
        print(f"Fetching window: offset {offset}...")
        resp = requests.get(url, headers={
            "x-rapidapi-host": AERODATABOX_HOST, 
            "x-rapidapi-key": AERODATABOX_KEY
        })
        resp.raise_for_status()
        data = json.loads(resp.text)
        master_data["arrivals"].extend(data.get('arrivals', []))
        master_data["departures"].extend(data.get('departures', []))
        
        # Respect API rate limits (1 request per second)
        time.sleep(1.1)

    # Save the combined JSON
    json_filename = f"flight_data_raw.json"
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(master_data, f, indent=4)
    
    print(f"Combined data saved to {json_filename}")

    with open("flight_data_raw.json", 'r') as f:
        data = json.load(f)

    def extract_times(flight_list, time_key):
        extracted = []
        for flight in flight_list:
            if flight.get('codeshareStatus') != 'IsOperator':
                continue
            movement = flight.get('movement', {})
            # We prioritize 'local' time as it already includes the offset
            scheduled = movement.get('scheduledTime', {}).get('local')
            revised = movement.get('revisedTime', {}).get('local')
            
            extracted.append({
                f'scheduled_{time_key}_times': scheduled,
                f'revised_{time_key}_times': revised
            })
        return pd.DataFrame(extracted)

    # Process Arrivals
    if 'arrivals' in data:
        df_arr = extract_times(data['arrivals'], 'arrival')
        df_arr.to_csv('arrivals.csv', index=False)
        print(f"✅ Saved {len(df_arr)} arrivals to data/arrivals.csv")

    # Process Departures
    if 'departures' in data:
        df_dep = extract_times(data['departures'], 'departure')
        df_dep.to_csv('departures.csv', index=False)
        print(f"✅ Saved {len(df_dep)} departures to data/departures.csv")

fetch_combined_flights()


Fetching window: offset -720...
Fetching window: offset 0...
Fetching window: offset 720...
Combined data saved to flight_data_raw.json
✅ Saved 1106 arrivals to data/arrivals.csv
✅ Saved 1149 departures to data/departures.csv
